### DuckDB Project
Sunny Yin

In [1]:
pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import duckdb

## Data Preparation

This section generates a synthetic sales dataset used for analytical queries.  
Users may alternatively provide their own dataset in CSV format with the required schema. (Set USE_SYNTHETIC = False)

### Create Synthetic Sales Database

In [3]:
USE_SYNTHETIC = True  # set to False to load your own dataset

if USE_SYNTHETIC:
    # default dataset size
    default_n = 2_000_000
    
    user_input = input(f"Enter number of rows (default {default_n}): ").strip()
    
    # use default if empty, otherwise convert to int
    n = int(user_input) if user_input else default_n

    products = ["Laptop", "Smartphone", "Headphones", "Monitor", "Keyboard",
                "Mouse", "Printer", "Speaker", "Webcam", "Router"]

    df = pd.DataFrame({
        "order_id": np.arange(1, n + 1),
        "order_date": pd.date_range("2026-01-01", periods=n, freq="min"),
        "region": np.random.choice(["US", "EU", "APAC", "LATAM"], n),
        "product": np.random.choice(products, n, p=[0.1, 0.2, 0.15, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.1]),
        "category": np.random.choice(["Electronics", "Home", "Office", "Sports"], n),
        "customer_id": np.random.randint(1000, 5000, n),
        "quantity": np.random.randint(1, 10, n),
        "unit_price": np.random.randint(10, 500, n),
    })

    df["sales"] = df["quantity"] * df["unit_price"]
    df.to_csv("sales_synthetic.csv", index=False)

    print(f"Synthetic dataset generated with {n:,} rows")

else:
    df = pd.read_csv("data/your_dataset.csv")
    print(f"Loaded user dataset with {len(df):,} rows")

Synthetic dataset generated with 2,000,000 rows


# DuckDB Processing

## Data Loading into DuckDB

This step loads the dataset into DuckDB as a relational table, and shows the table schema.

In [4]:
con = duckdb.connect("sales.duckdb")

con.execute("""
CREATE OR REPLACE TABLE sales_synthetic AS
SELECT * FROM read_csv_auto('sales_synthetic.csv')
""")

In [5]:
con.execute("SELECT COUNT(*) FROM sales_synthetic").fetchall()

[(2000000,)]

In [6]:
con.execute("DESCRIBE sales_synthetic").fetchdf()

,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,order_date,TIMESTAMP,YES,None,None,None
2,region,VARCHAR,YES,None,None,None
3,product,VARCHAR,YES,None,None,None
4,category,VARCHAR,YES,None,None,None
5,customer_id,BIGINT,YES,None,None,None
6,quantity,BIGINT,YES,None,None,None
7,unit_price,BIGINT,YES,None,None,None
8,sales,BIGINT,YES,None,None,None


In [7]:
con.execute("SELECT * FROM sales_synthetic LIMIT 5").fetchdf()

,order_id,order_date,region,product,category,customer_id,quantity,unit_price,sales
0,1,2026-01-01 00:00:00,LATAM,Webcam,Office,4362,3,200,600
1,2,2026-01-01 00:01:00,US,Webcam,Sports,2764,4,319,1276
2,3,2026-01-01 00:02:00,EU,Monitor,Sports,2313,5,384,1920
3,4,2026-01-01 00:03:00,APAC,Smartphone,Home,4292,2,63,126
4,5,2026-01-01 00:04:00,US,Headphones,Sports,4439,7,38,266


## Aggregation: Sales by Region

This query computes total sales grouped by region and uses `EXPLAIN` to inspect DuckDB’s physical execution plan.

In [8]:
query = """
SELECT region, SUM(sales) AS total_sales
FROM sales_synthetic
GROUP BY region
ORDER BY region
"""

result = con.execute(query).fetchdf()
result

,region,total_sales
0,APAC,637520460.0
1,EU,635715818.0
2,LATAM,635988538.0
3,US,637535228.0


In [9]:
plan = con.execute("""
EXPLAIN SELECT region, SUM(sales)
FROM sales_synthetic
GROUP BY region
""").fetchall()

print(plan[0][1])

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│          ~4 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│    sum_no_overflow(#1)    │
│                           │
│          ~4 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│           region          │
│           sales           │
│                           │
│      ~2,000,000 rows      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        ubigint(#0)        │
│             #1            │
│         

## Query Plan Analysis (EXPLAIN ANALYZE)

This section uses `EXPLAIN ANALYZE` to examine how DuckDB executes the aggregation query internally.

The output includes:
- **Total execution time**, showing overall query performance
- A **physical operator tree**, illustrating how data flows through the execution pipeline
- **Operator-level statistics**, including number of rows processed and execution time

This allows us to observe key internal behaviors such as:
- **Sequential scan over columnar storage**
- **Vectorized processing using DataChunks**
- **Hash-based aggregation (HASH_GROUP_BY)**
- **Push-based execution pipeline**
- Presence of **pipeline breakers** (e.g., aggregation)

The results provide direct insight into how DuckDB processes large datasets efficiently.

In [10]:
plan = con.execute("""
EXPLAIN ANALYZE SELECT region, SUM(sales)
FROM sales_synthetic
GROUP BY region
""").fetchall()

print(plan[0][1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT region, SUM(sales) FROM sales_synthetic GROUP BY region 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0075s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             

## Filtering + Aggregation: Conditional Sales Analysis

This query computes total sales by region for a filtered subset of the data.

The filter condition restricts the dataset based on:
- `order_date` (time-based filtering)
- `product` (categorical filtering)

This demonstrates how DuckDB applies **filter pushdown**, allowing irrelevant data to be discarded early during the scan phase. As a result, fewer rows are processed in the aggregation step, improving performance.

In [11]:
result_filtered = con.execute("""
SELECT region, SUM(sales) AS total_sales
FROM sales_synthetic
WHERE  order_date >= '2026-03-04' and product IN ('Laptop', 'Smartphone')
GROUP BY region
""").fetchdf()

result_filtered

,region,total_sales
0,APAC,182094247.0
1,LATAM,181786602.0
2,EU,182001405.0
3,US,183127743.0


In [12]:
result_filtered = con.execute("""
EXPLAIN SELECT region, SUM(sales) AS total_sales
FROM sales_synthetic
WHERE  order_date >= '2026-03-04' and product IN ('Laptop', 'Smartphone')
GROUP BY region
""").fetchall()

print(result_filtered[0][1])

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│          ~4 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│    sum_no_overflow(#1)    │
│                           │
│          ~4 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│           region          │
│           sales           │
│                           │
│        ~80,000 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        ubigint(#0)        │
│             #1            │
│         

## Filtering (Single-Value Aggregation)

This query computes total sales for a **specific subset of data**, filtered by:
- `region = 'US'`
- `order_date >= '2026-03-04'`
- `product IN ('Laptop', 'Smartphone')`

Unlike the previous aggregation, this query does not use `GROUP BY` and instead returns a **single aggregated value**.

This example highlights how DuckDB:
- Applies **filter pushdown** during the scan phase
- Processes only a small subset of data before aggregation

As a result, the query executes efficiently by minimizing both I/O and computation.

In [13]:
result_filtered_1 = con.execute("""
SELECT SUM(sales) AS total_sales
FROM sales_synthetic
WHERE region = 'US'
  AND order_date >= '2026-03-04'
  AND product IN ('Laptop', 'Smartphone')
""").fetchdf()

result_filtered_1

,total_sales
0,183127743.0


In [14]:
result_filtered_1 = con.execute("""
EXPLAIN SELECT SUM(sales) AS total_sales
FROM sales_synthetic
WHERE region = 'US'
  AND order_date >= '2026-03-04'
  AND product IN ('Laptop', 'Smartphone')
""").fetchall()

print(result_filtered_1[0][1])

┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│    sum_no_overflow(#0)    │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│           sales           │
│                           │
│       ~100,000 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│             #1            │
│                           │
│       ~100,000 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ ((product = 'Laptop') OR  │
│ (product = 'Smartphone')) │
│                           │
│       ~100,000 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          SEQ_SCAN         │
│    ────────────────────   │
│           Table:          │
│ sales.main.sales_synthetic│
│         

## Ranking: Top-N Products by Sales

This query identifies the top-performing products based on total sales.

It combines:
- **Aggregation** (`GROUP BY product`)
- **Sorting** (`ORDER BY total_sales DESC`)
- **Limiting** (`LIMIT 10`)

This demonstrates how DuckDB handles **multi-stage analytical queries**, where results are first aggregated and then ranked.

From an internal perspective, this query highlights:
- **Vectorized aggregation** using `HASH_GROUP_BY`
- A **TOP-N operator**, which efficiently combines sorting and limiting
- The presence of **pipeline breakers** (aggregation and sorting), which require partial materialization of results
- Reduced computation by performing sorting on already aggregated data instead of raw rows

This pattern is common in analytical workloads where only a small subset of top results is needed.

In [15]:
result_top = con.execute("""
SELECT product, SUM(sales) AS total_sales
FROM sales_synthetic
GROUP BY product
ORDER BY total_sales DESC
LIMIT 10
""").fetchdf()

result_top

,product,total_sales
0,Smartphone,508849835.0
1,Headphones,382628627.0
2,Keyboard,255052423.0
3,Laptop,254609815.0
4,Mouse,254461461.0
5,Router,254181532.0
6,Monitor,253685318.0
7,Speaker,128085197.0
8,Printer,127863230.0
9,Webcam,127342606.0


In [16]:
# explain
result_top = con.execute("""
EXPLAIN SELECT product, SUM(sales) AS total_sales
FROM sales_synthetic
GROUP BY product
ORDER BY total_sales DESC
LIMIT 10
""").fetchall()

print(result_top[0][1])

┌───────────────────────────┐
│           TOP_N           │
│    ────────────────────   │
│          Top: 10          │
│                           │
│         Order By:         │
│       sum(sales.main      │
│  .sales_synthetic.sales)  │
│            DESC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│          ~10 rows         │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│    sum_no_overflow(#1)    │
│                           │
│          ~10 rows         │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│          product          │
│         

In [17]:
# by count instead of sum
result_top_count = con.execute("""
SELECT product, COUNT(*) AS order_count 
FROM sales_synthetic
GROUP BY product
ORDER BY order_count DESC
LIMIT 10
""").fetchdf()

result_top_count

,product,order_count
0,Smartphone,399943
1,Headphones,300300
2,Keyboard,200284
3,Router,199971
4,Laptop,199789
5,Mouse,199464
6,Monitor,199355
7,Speaker,100678
8,Webcam,100212
9,Printer,100004


In [18]:
# explore top products by count instead of sum
result_top_count = con.execute("""
EXPLAIN SELECT product, COUNT(*) AS order_count
FROM sales_synthetic
GROUP BY product
ORDER BY order_count DESC
LIMIT 10
""").fetchall()

print(result_top_count[0][1])

┌───────────────────────────┐
│           TOP_N           │
│    ────────────────────   │
│          Top: 10          │
│                           │
│         Order By:         │
│     count_star() DESC     │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│          ~10 rows         │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│        count_star()       │
│                           │
│          ~10 rows         │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│          product          │
│                           │
│      ~2,000,000 rows      │
└─────────

## Join + Aggregation: Category-Level Sales Analysis （Extra feature）

This query joins the large `sales_synthetic` table with a small derived `products` table to map each product to a category, and then computes total sales by category.

From the execution plan:
- DuckDB performs a **HASH_JOIN**, building a hash table from the small `products` table (~10 rows) and probing it with the large `sales_synthetic` table (~2,000,000 rows)
- Data is processed in **vectorized batches (DataChunks)** through sequential scans
- A **HASH_GROUP_BY** operator aggregates sales by category
- A **TOP_N** operator efficiently returns the top results without fully sorting all data

This demonstrates how DuckDB efficiently combines **join, aggregation, and ranking** in a single analytical query while minimizing overhead on large datasets.

In [19]:
con.execute("""
CREATE OR REPLACE TABLE products AS
SELECT DISTINCT product, 
       CASE 
         WHEN product IN ('Laptop','Smartphone') THEN 'Electronics'
         ELSE 'Other'
       END AS category
FROM sales_synthetic
""")

In [20]:
result_join = con.execute("""
SELECT p.category, SUM(s.sales) AS total_sales
FROM sales_synthetic s
JOIN products p
ON s.product = p.product
GROUP BY p.category
ORDER BY total_sales DESC
LIMIT 5
""").fetchdf()

result_join

,category,total_sales
0,Other,1.783300e+09
1,Electronics,7.634596e+08


In [21]:
# explain the result_join
result_join_explain = con.execute("""
EXPLAIN SELECT p.category, SUM(s.sales) AS total_sales
FROM sales_synthetic s
JOIN products p
ON s.product = p.product
GROUP BY p.category
ORDER BY total_sales DESC
LIMIT 5
""").fetchall()

print(result_join_explain[0][1])

┌───────────────────────────┐
│           TOP_N           │
│    ────────────────────   │
│           Top: 5          │
│                           │
│         Order By:         │
│     sum(s.sales) DESC     │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│        ~2,277 rows        │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│    Aggregates: sum(#1)    │
│                           │
│        ~2,277 rows        │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│          category         │
│           sales           │
│                           │
│      ~2,000,000 rows      │
└─────────────┬─────────────┘
┌─────────

## Performance Comparison: DuckDB vs Pandas

This section compares the execution time of the same aggregation query using DuckDB and Pandas.

- **DuckDB** executes the query using its vectorized, columnar execution engine.
- **Pandas** performs the equivalent operation using in-memory DataFrame processing.

The goal is to demonstrate how DuckDB’s internal optimizations—such as **vectorized execution, efficient columnar scans, and reduced I/O**—can lead to faster performance for analytical workloads.

Execution time is measured for both approaches to provide a direct comparison.

In [22]:
import time

start = time.time()

con.execute("""
SELECT region, SUM(sales)
FROM sales_synthetic
GROUP BY region
""").fetchdf()

end = time.time()

print("Execution time in ms:", (end - start) * 1000)

Execution time in ms: 4.903078079223633


In [23]:

df = pd.read_csv("sales_synthetic.csv")

%time df.groupby("region")["sales"].sum()

CPU times: total: 78.1 ms
Wall time: 74.2 ms


region
APAC     637520460
EU       635715818
LATAM    635988538
US       637535228
Name: sales, dtype: int64

**Observation:** DuckDB performs significantly faster than Pandas for this aggregation due to its optimized query execution engine.